In [ ]:
!pip install -q transformers datasets evaluate torch setfit wandb sentence-transformers python-dotenv peft

In [ ]:
import json
import logging
import random
import torch
from dataclasses import dataclass
import wandb
import os
import evaluate
from datasets import Dataset, ClassLabel, DatasetDict, load_dataset
from peft import LoraConfig, get_peft_model, TaskType
from transformers import (
    pipeline,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)
import numpy as np
from torch.utils.data import WeightedRandomSampler
from sklearn.utils.class_weight import compute_class_weight
import numpy as np


In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger(__name__)

In [ ]:
@dataclass
class TrainingConfig:
    model_name: str = "microsoft/deberta-v3-small"
    zeroshot_model: str = "facebook/bart-large-mnli"
    setfit_model: str = "sentence-transformers/paraphrase-mpnet-base-v2"
    output_dir: str = "/tmp/model_finetuned/"
    hf_repo: str = "kentokamg/ticket-triage-finetuned"
    epochs: int = 5
    batch_size: int = 16
    lr: float = 2e-5
    max_length: int = 256
    test_size: float = 0.2
    seed: int = 42
    few_shot_n: int = 8
    zeroshot_batch: int = 32
    wandb_project: str = "ticket-triage"
    lora_r: int = 8
    lora_alpha: int = 16
    lora_dropout: float = 0.1

cfg = TrainingConfig()
logger.info(f"Output dir: {cfg.output_dir}")

In [ ]:
wandb.login()
wandb.init(
    project=cfg.wandb_project,
    job_type="fine-tuning",
    config={
        "model": cfg.model_name,
        "epochs": cfg.epochs,
        "batch_size": cfg.batch_size,
        "learning_rate": cfg.lr,
        "max_length": cfg.max_length,
    }
)

In [ ]:
CATEGORIES = ["biling", "technical", "shipping", "account", "general"]

f1_metric = evaluate.load('f1')
acc_metric = evaluate.load('accuracy')

In [ ]:
def load_dataset_from_hf() -> DatasetDict:
    ds = load_dataset('kentokamg/ticket-dataset')

    label_names = sorted(set(ds['train']['category']))
    label_feature = ClassLabel(names=label_names)
    id2label = {i: label for i, label in enumerate(label_names)}
    label2id = {label: i for i, label in enumerate(label_names)}
    ds = ds['train'].cast_column('category', label_feature)
    ds = ds.train_test_split(test_size=cfg.test_size, stratify_by_column='category', seed=cfg.seed)
    return ds, id2label, label2id

In [ ]:
def tokenize_dataset(hf_dataset, tokenizer):
    cols_to_remove = [c for c in hf_dataset["train"].column_names if c != "category"]

    def tokenize_function(batch):
        tokens = tokenizer(
            batch["text"],
            truncation=True,
            max_length=cfg.max_length,
        )
        tokens["labels"] = batch["category"]
        return tokens

    return hf_dataset.map(tokenize_function, batched=True, remove_columns=cols_to_remove)

In [ ]:
def compute_metrics(eval_pred) -> dict:
    f1_metric = evaluate.load('f1')
    acc_metric = evaluate.load('accuracy')
    
    logits, labels = eval_pred
    predictions    = np.argmax(logits, axis=-1)
 
    f1  = f1_metric.compute(predictions=predictions, references=labels, average="macro")
    acc = acc_metric.compute(predictions=predictions, references=labels)
 
    f1_per_class = f1_metric.compute(
        predictions=predictions,
        references=labels,
        average=None,
        labels=list(range(len(CATEGORIES))),
    )
 
    metrics = {"f1_macro": f1["f1"], "accuracy": acc["accuracy"]}
    for i, cat in enumerate(CATEGORIES):
        metrics[f"f1_{cat}"] = float(f1_per_class["f1"][i])
 
    return metrics

In [ ]:
def zero_shot(ds: DatasetDict) -> dict:
    logger.info(f'Evaluando zero-shot con {cfg.zeroshot_model}')
    classifier = pipeline('zero-shot-classification',
                          model=cfg.zeroshot_model,
                          device=0 if device == 'cuda' else -1,
                          batch_size=cfg.zeroshot_batch)
    
    test_texts = list(ds['test']['text'])
    test_labels = list(ds['test']['category']) 

    results = classifier(
        test_texts,
        candidate_labels=CATEGORIES,
        multi_label=False
    )
    
    f1_metric = evaluate.load('f1')
    acc_metric = evaluate.load('accuracy')

    predictions = [CATEGORIES.index(r['labels'][0]) for r in results]
    f1 = f1_metric.compute(predictions=predictions, references=test_labels, average='macro')
    accuracy = acc_metric.compute(predictions=predictions, references=test_labels)

    wandb.log({
        'zero-shot/f1_macro': f1['f1'],
        'zero-shot/accuracy': accuracy['accuracy']
    })
    logger.info(f"Zero-shot — F1: {f1['f1']:.3f} | Acc: {accuracy['accuracy']:.3f}")
    return {"f1": f1["f1"], "acc": accuracy["accuracy"]}


In [ ]:
def finetune(ds: DatasetDict, id2label, label2id, tokenizer, tokenized_ds) -> dict:
    logger.info(f"Fine-tuning {cfg.model_name} con LoRA (r={cfg.lora_r}, alpha={cfg.lora_alpha})")
    tokenizer     = tokenizer
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    train_labels = tokenized_ds['train']['labels']
    class_weights = compute_class_weight('balanced',
                                         classes = np.unique(train_labels),
                                         y=train_labels)
    class_weights = torch.tensor(class_weights, dtype=torch.float)
    sample_weights = class_weights[train_labels]
    sampler = WeightedRandomSampler(
                                    weights=sample_weights,
                                    num_samples=len(sample_weights),
                                    replacement=True)

    model = AutoModelForSequenceClassification.from_pretrained(
        cfg.model_name,
        num_labels=len(id2label),
        id2label=id2label,
        label2id=label2id,
    )

    lora_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=cfg.lora_r,
        lora_alpha=cfg.lora_alpha,
        lora_dropout=cfg.lora_dropout,
        target_modules=["query_proj", "key_proj", "value_proj"],
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    training_args = TrainingArguments(
        output_dir=cfg.output_dir,
        num_train_epochs=cfg.epochs,
        per_device_train_batch_size=cfg.batch_size,
        per_device_eval_batch_size=cfg.batch_size,
        learning_rate=cfg.lr,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        report_to="wandb",
        logging_steps=50,
        seed=cfg.seed,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_ds["train"],
        eval_dataset=tokenized_ds["test"],
        compute_metrics=compute_metrics,
        data_collator=data_collator,
        train_sampler=sampler
    )

    trainer.train()

    final_metrics = trainer.evaluate()
    f1  = final_metrics.get("eval_f1_macro", 0)
    acc = final_metrics.get("eval_accuracy", 0)
    logger.info(f"Fine-tuned (LoRA) — F1: {f1:.3f} | Accuracy: {acc:.3f}")

    wandb.log({"finetuned/f1_macro": f1, "finetuned/accuracy": acc})

    trainer.save_model(cfg.output_dir)
    tokenizer.save_pretrained(cfg.output_dir)

    return {"f1": f1, "acc": acc, "trainer": trainer, "model": model, "tokenizer": tokenizer}

In [ ]:
from huggingface_hub import login

login()

finetuned_results = finetune(ds)

logger.info(f"Subiendo modelo a HF Hub: {cfg.hf_repo}...")
finetuned_results["model"].push_to_hub(cfg.hf_repo, private=False)
finetuned_results["tokenizer"].push_to_hub(cfg.hf_repo, private=False)
logger.info("Modelo subido a HuggingFace Hub")

In [ ]:
def log_comparison(zero_shot: dict, finetuned: dict):
 
    systems = [
        ("1. Zero-shot (BART-MNLI)",    zero_shot),
        ("3. Fine-tuned (DeBERTa)",      finetuned),
    ]
 
    table = wandb.Table(columns=["Sistema", "F1 Macro", "Accuracy"])
    for name, results in systems:
        table.add_data(name, round(results["f1"], 3), round(results["acc"], 3))
 
    wandb.log({
        "comparison/table":        table,
        "comparison/gain_vs_zero": finetuned["f1"] - zero_shot["f1"],
    })

    logger.info("COMPARATIVA FINAL")
    for name, results in systems:
        logger.info(f"  {name:35} F1: {results['f1']:.3f} | Acc: {results['acc']:.3f}")
    logger.info(f"\n  Mejora vs zero-shot:  +{finetuned['f1'] - zero_shot['f1']:.3f}")
    logger.info("=" * 55)

In [ ]:
def main():
    random.seed(cfg.seed)

    wandb.init(
        project=cfg.wandb_project,
        job_type="comparativa",
        config=cfg.__dict__,
    )
    logger.info("Cargando dataset desde HuggingFace Hub")
    ds, id2label, label2id = load_dataset_from_hf()
    zero_shot_results  = zero_shot(ds)
    
    tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)
    tokenized_ds = tokenize_dataset(ds, tokenizer)
    logger.info(f"Dataset tokenizado: {tokenized_ds}")
    
    finetuned_results = finetune(ds, id2label, label2id, tokenizer, tokenized_ds)
    log_comparison(zero_shot_results, finetuned_results)

    wandb.finish()


if __name__ == "__main__":
    main()